# Evaluación externa de EfficientNet-V2-S en HYGD

Este cuaderno evalúa un modelo entrenado para priorizar la revisión de retinografías con signos compatibles con glaucoma. La evaluación incluye 731 imágenes del Hillel Yaffe Glaucoma Dataset (HYGD) y utiliza el umbral fijado durante la validación interna.

## Archivos necesarios

En la primera celda de código, ajusta `root` para que apunte a la carpeta del proyecto que contiene `04_Kaggle_2` y `05_Evaluacion_externa`. El cuaderno necesita los siguientes archivos:

- `05_Evaluacion_externa/hillel-yaffe-glaucoma-dataset-1.1.0.zip` — archivo original de HYGD, disponible en [PhysioNet](https://doi.org/10.13026/m92s-0z95).
- `05_Evaluacion_externa/auditoria_visual_HYGD_v1/` — debe contener `protocolo.json` y `cohorte_fijada.csv`, generados durante la auditoría de las imágenes.
- `04_Kaggle_2/resultados/exp02_corregido/` — debe contener `config.json`, `decision_locked.json` y el archivo de pesos entrenados cuyo nombre aparece en `decision_locked.json`.

Ejecuta las tres celdas de código en orden. Puedes utilizar CPU. Las predicciones, las métricas y el histograma de puntuaciones se guardan en `05_Evaluacion_externa/evaluacion_HYGD_umbral_original/`.


## 1. Verificar el protocolo de evaluación

Antes de calcular las predicciones, esta celda comprueba que la cohorte, los pesos del modelo, la configuración y el umbral coinciden con los fijados para la evaluación externa.

El resultado esperado es de 731 retinografías correspondientes a 284 identificadores de paciente. Según las etiquetas de referencia de HYGD, 536 imágenes son positivas y 195 son negativas.

In [ ]:
from pathlib import Path
import hashlib
import json
import pandas as pd

# Set this to the folder containing 04_Kaggle_2 and 05_Evaluacion_externa.
root = Path.cwd()

external = root / '05_Evaluacion_externa'
audit = external / 'auditoria_visual_HYGD_v1'
run = root / '04_Kaggle_2/resultados/exp02_corregido'
zip_path = external / 'hillel-yaffe-glaucoma-dataset-1.1.0.zip'

def sha256_file(path):
    with path.open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

protocol = json.loads((audit / 'protocolo.json').read_text())
decision = json.loads((run / 'decision_locked.json').read_text())
cohort_path = audit / 'cohorte_fijada.csv'
checkpoint = run / decision['weights_file']
assert sha256_file(cohort_path) == protocol['cohorte_sha256']
assert sha256_file(checkpoint) == decision['weights_sha256'] == protocol['pesos_sha256']
assert sha256_file(run / 'config.json') == decision['config_sha256']
config = json.loads((run / 'config.json').read_text())
assert config['image_size'] == 224
assert decision['positive_target'] == 1 and decision['comparison'] == 'score >= threshold'
assert decision['threshold'] == protocol['threshold'] == 0.8970574736595154

cohort = pd.read_csv(cohort_path)
assert len(cohort) == 731 and cohort.target.value_counts().to_dict() == {1: 536, 0: 195}
assert cohort.filename.is_unique and cohort.patient.nunique() == 284

print(len(cohort), int(cohort.target.sum()), cohort.patient.nunique())

## 2. Obtener una puntuación por imagen

Lee las retinografías en lotes de 16, verifica que coincidan con la cohorte y aplica la misma preparación que en el piloto. Deben guardarse 731 filas en predicciones.csv.

In [ ]:
from zipfile import ZipFile
from io import BytesIO
from PIL import Image, ImageOps
import torch
from torch import nn
from torchvision import transforms as T
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

class SquarePad:
    def __call__(self, im):
        w, h = im.size
        side = max(w, h)
        left, top = (side-w)//2, (side-h)//2
        return ImageOps.expand(im, (left, top, side-w-left, side-h-top), fill=0)

preset = EfficientNet_V2_S_Weights.IMAGENET1K_V1.transforms()
transform = T.Compose([
    SquarePad(),
    T.Resize((224,224), interpolation=T.InterpolationMode.BILINEAR),
    T.ToTensor(),
    T.Normalize(preset.mean, preset.std),
])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = efficientnet_v2_s(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model.load_state_dict(torch.load(checkpoint, map_location='cpu', weights_only=True), strict=True)
model.to(device).eval()
print('Pesos cargados.', device)
import shutil
from tqdm.auto import tqdm
from tempfile import TemporaryDirectory

output = external / 'evaluacion_HYGD_umbral_original'
output.mkdir(exist_ok=True)
batches = []
with TemporaryDirectory() as temp:
    local_zip = shutil.copyfile(zip_path, Path(temp) / 'hygd.zip')
    with ZipFile(local_zip) as archive, torch.inference_mode():
        label_file, = [x for x in archive.namelist() if x.endswith('/Labels.csv')]
        labels = pd.read_csv(BytesIO(archive.read(label_file))).set_index('Image Name', verify_integrity=True)
        members = [x for x in archive.namelist() if '/Images/' in x and x.lower().endswith('.jpg')]
        paths = {Path(x).name: x for x in members}
        assert len(paths) == len(members)
        for start in tqdm(range(0, len(cohort), 16)):
            batch = cohort.iloc[start:start+16].copy()
            images, names = [], []
            for row in batch.itertuples(index=False):
                member = paths[row.filename]
                name = Path(member).name
                reference = labels.loc[name]
                assert {'GON+':1, 'GON-':0}[reference['Label']] == row.target
                assert int(reference['Patient']) == row.patient
                with Image.open(BytesIO(archive.read(member))) as original:
                    rgb = original.convert('RGB')
                    digest = hashlib.sha256(str(rgb.size).encode() + rgb.tobytes()).hexdigest()
                    assert digest == row.pixel_sha256, name
                    images.append(transform(rgb))
                names.append(name)
            assert names == batch.filename.tolist()
            x = torch.stack(images)
            assert x.shape == (len(batch),3,224,224) and torch.isfinite(x).all()
            logits = model(x.to(device))
            assert logits.shape == (len(batch),2) and torch.isfinite(logits).all()
            batch['score'] = logits.softmax(dim=1)[:,1].cpu().numpy().astype('float64')
            batches.append(batch)
pred = pd.concat(batches, ignore_index=True)
assert pred[cohort.columns].equals(cohort) and pred.score.between(0,1).all()
pred['priority'] = pred.score >= decision['threshold']
pred.to_csv(output / 'predicciones.csv', index=False)
(output / 'config_ejecucion.json').write_text(json.dumps({
    'decision': decision, 'protocolo': protocol, 'device': str(device),
    'torch': str(torch.__version__), 'predicciones_sha256': sha256_file(output / 'predicciones.csv'),
}, indent=2, ensure_ascii=False))
print('Predicciones guardadas:', len(pred))

## 3. Calcular resultados

Calcula la capacidad de ordenar (AUROC y AP) y lo que sucede con el umbral original. Los intervalos agrupan las imágenes de un mismo paciente. Debe aparecer EVALUACIÓN OK.

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
import matplotlib.pyplot as plt

def metrics(data):
    y, score, selected = data.target, data.score, data.priority
    tn, fp, fn, tp = map(int, confusion_matrix(y, selected, labels=[0,1]).ravel())
    k = tp + fp
    expected = k * (tp + fn) / len(data)
    return {
        'AUROC': roc_auc_score(y, score) if y.nunique() == 2 else np.nan,
        'AP': average_precision_score(y, score) if y.nunique() == 2 else np.nan,
        'sensibilidad': tp / (tp + fn) if tp + fn else np.nan,
        'especificidad': tn / (tn + fp) if tn + fp else np.nan,
        'fraccion_priorizada': k / len(data),
        'priorizadas': k, 'TN': tn, 'FP': fp, 'FN': fn, 'TP': tp,
        'positivas_esperadas_al_azar': expected,
        'diferencia_frente_al_azar': tp - expected,
    }

point = metrics(pred)
groups = list(pred.groupby('patient', sort=True).indices.values())
rng = np.random.default_rng(42)
draws = []
for _ in tqdm(range(2000), desc='Remuestreo por paciente'):
    ids = rng.integers(0, len(groups), len(groups))
    rows = np.concatenate([groups[i] for i in ids])
    draws.append(metrics(pred.iloc[rows]))
boot = pd.DataFrame(draws)
columns = ['AUROC','AP','sensibilidad','especificidad','fraccion_priorizada','diferencia_frente_al_azar']
intervals = {
    col: {'inferior': float(boot[col].quantile(.025)), 'superior': float(boot[col].quantile(.975)),
          'replicas_validas': int(boot[col].notna().sum())}
    for col in columns
}
summary = {'n': len(pred), 'positivas': int(pred.target.sum()), 'pacientes': len(groups),
           'umbral': decision['threshold'], 'metricas': point, 'IC95': intervals}
(output / 'metricas.json').write_text(json.dumps(summary, indent=2, allow_nan=False))
fig, ax = plt.subplots(figsize=(8,4))
for target, label in [(0,'GON−'), (1,'GON+')]:
    ax.hist(pred.loc[pred.target == target, 'score'], bins=np.linspace(0,1,26),
            weights=np.ones(int((pred.target == target).sum())) / int((pred.target == target).sum()),
            alpha=.55, label=label)
ax.axvline(decision['threshold'], color='black', linestyle='--', label='Umbral original')
ax.set(xlabel='Puntuación de clase 1', ylabel='Fracción dentro de cada clase', xlim=(0,1))
ax.legend()
fig.tight_layout()
fig.savefig(output / 'puntuaciones_HYGD.png', dpi=180)
plt.show()
display(pd.Series(point))
print('EVALUACIÓN OK | 731 retinografías')